<a href="https://colab.research.google.com/github/K4ws/Movimento-Browniano-cont-nuo-/blob/main/Motion_Brownian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. COLETA DE DADOS
# ==========================================
ativos = ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'WEGE3.SA', 'MBRF3.SA']

n_ativos = len(ativos)
pesos = np.random.random(n_ativos)
pesos /= np.sum(pesos)

data_inicio = '2021-01-01'
data_fim = '2024-05-01'

print("Baixando dados históricos...")
dados = yf.download(ativos, start=data_inicio, end=data_fim)['Close']
dados = dados.ffill().dropna()

# ==========================================
# 2. CALIBRAÇÃO ESTATÍSTICA (EXTRAINDO SALTOS)
# ==========================================
retornos_diarios = np.log(dados / dados.shift(1)).dropna()
dias_uteis_ano = 252
S0 = dados.iloc[-1].values

def calibrar_parametros_merton(retornos, limite_std=3.0):
    """
    Separa o Movimento Browniano contínuo dos Saltos (Cisnes Negros)
    usando um filtro de 3 desvios padrões (3-Sigma Rule).
    """
    mu_normal, sigma_normal = [], []
    lambda_j, mu_j, sigma_j = [], [], []

    print("\n--- CALIBRAÇÃO DOS SALTOS (DADOS HISTÓRICOS) ---")

    for ativo in retornos.columns:
        ret = retornos[ativo]
        media_bruta = ret.mean()
        std_bruto = ret.std()

        # Filtro: Dias onde o retorno excedeu 3 desvios padrões
        is_jump = (ret > media_bruta + limite_std * std_bruto) | (ret < media_bruta - limite_std * std_bruto)

        dias_normais = ret[~is_jump]
        dias_salto = ret[is_jump]

        # 1. Parâmetros Brownianos (Anualizados, baseados apenas nos dias normais)
        mu_normal.append(dias_normais.mean() * dias_uteis_ano)
        sigma_normal.append(dias_normais.std() * np.sqrt(dias_uteis_ano))

        # 2. Parâmetros de Salto
        qtd_saltos = len(dias_salto)
        if qtd_saltos > 0:
            freq_anual = (qtd_saltos / len(ret)) * dias_uteis_ano
            lambda_j.append(freq_anual)
            mu_j.append(dias_salto.mean()) # Tamanho diário do salto
            sigma_j.append(dias_salto.std() if qtd_saltos > 1 else 0.0)
        else:
            lambda_j.append(0.0)
            mu_j.append(0.0)
            sigma_j.append(0.0)
            freq_anual = 0

        print(f"{ativo}: Encontrados {qtd_saltos} eventos extremos (~{freq_anual:.1f} por ano)")

    print("------------------------------------------------\n")

    return (np.array(mu_normal), np.array(sigma_normal),
            np.array(lambda_j), np.array(mu_j), np.array(sigma_j))

# Extraindo os vetores calibrados
mu, sigma, lambda_salto, mu_salto, sigma_salto = calibrar_parametros_merton(retornos_diarios)

# Correlação (calculada sobre todos os dados)
df_correlacao = retornos_diarios.corr()
corr_matrix = df_correlacao.values

# ==========================================
# 3. SIMULAÇÃO: MERTON JUMP DIFFUSION
# ==========================================
def simular_jump_diffusion_auto(S0, mu, sigma, corr_matrix, lambda_j, mu_j, sigma_j, pesos, T, passos, n_trajetorias):
    n_ativos = len(S0)
    dt = T / passos
    L = np.linalg.cholesky(corr_matrix)

    # 1. COMPONENTE BROWNIANO
    z_independente = np.random.standard_normal((n_trajetorias, passos, n_ativos))
    z_correlacionado = np.matmul(z_independente, L.T)

    drift = (mu - 0.5 * sigma**2) * dt
    difusao = sigma * np.sqrt(dt)
    retornos_brownianos = drift + difusao * z_correlacionado

    # 2. COMPONENTE DE SALTOS (Dinâmico por Ativo)
    # O Poisson agora usa um vetor lambda_j, criando frequências diferentes para cada empresa
    ocorrencia_saltos = np.random.poisson(lam=lambda_j * dt, size=(n_trajetorias, passos, n_ativos))
    tamanho_saltos = np.random.normal(loc=mu_j, scale=sigma_j, size=(n_trajetorias, passos, n_ativos))

    retornos_saltos = ocorrencia_saltos * tamanho_saltos

    # 3. UNIÃO DOS PROCESSOS
    retornos_totais = retornos_brownianos + retornos_saltos

    caminho_retornos = np.exp(np.cumsum(retornos_totais, axis=1))
    caminho_retornos = np.concatenate([np.ones((n_trajetorias, 1, n_ativos)), caminho_retornos], axis=1)

    valor_inicial = 10000.0
    valor_carteira = np.sum(caminho_retornos * pesos, axis=2) * valor_inicial

    return valor_carteira.T

# Execução
cenarios = 10000
passos_futuros = 252
print("Executando Simulação com Difusão de Saltos Calibrada...")
simulacoes = simular_jump_diffusion_auto(S0, mu, sigma, corr_matrix, lambda_salto, mu_salto, sigma_salto, pesos, 1.0, passos_futuros, cenarios)

# ==========================================
# 4. ANÁLISE E VISUALIZAÇÃO DE RISCO
# ==========================================
valores_finais = simulacoes[-1, :]
var_95 = np.percentile(valores_finais, 5)
cvar_95 = valores_finais[valores_finais <= var_95].mean()

plt.figure(figsize=(12, 7))
plt.plot(simulacoes[:, :100], color='blue', alpha=0.05)
plt.plot(np.mean(simulacoes, axis=1), color='red', linewidth=2, label='Média Esperada')

plt.fill_between(range(passos_futuros+1),
                 np.percentile(simulacoes, 5, axis=1),
                 np.percentile(simulacoes, 95, axis=1),
                 color='gray', alpha=0.2, label='Intervalo 90% Confiança')

plt.title(f"Merton Jump Diffusion (Parâmetros Extraídos do Mercado) - {cenarios} Cenários")
plt.ylabel("Valor da Carteira (R$)")
plt.xlabel("Dias Úteis (1 Ano)")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

print(f"\n--- RESULTADOS DA SIMULAÇÃO (Daqui a 1 Ano) ---")
print(f"VaR 95%: R$ {10000 - var_95:.2f} (Perda máxima em 95% dos casos)")
print(f"CVaR 95%: R$ {10000 - cvar_95:.2f} (Perda média se o VaR for rompido)")